# Proyecto 1 — Búsqueda en espacios de estados
## Problema 4: Caballo de Ajedrez — Variante (a)
### Camino mínimo en tablero 8×8

**Curso:** Inteligencia Artificial  
**Librería:** [baile](https://github.com/kyriox/baile)  

**Integrantes:**
- Ivan Montoya
- Gustavo Perez
- Johan Cruz

**Enlace al fork:** [https://github.com/00Gus/baile](https://github.com/00Gus/baile)


## 1. Descripción del problema

En el ajedrez, el **caballo** se mueve en forma de **"L"**: avanza dos casillas en una dirección
(horizontal o vertical) y una casilla en la dirección perpendicular. Esto le da hasta **8 movimientos
posibles** desde una casilla central, aunque menos desde los bordes y esquinas.

### Variante (a): Camino mínimo

Dado un tablero de **8×8**, una **casilla de inicio** y una **casilla meta**, el objetivo es encontrar
la secuencia de saltos del caballo que lo lleve del inicio a la meta con el **menor número de
movimientos posible**.

Este problema tiene un espacio de estados pequeño (64 casillas) pero es ideal para comparar
limpiamente el comportamiento y rendimiento de las estrategias BFS, DFS y A*.

### Movimientos del caballo

Los 8 desplazamientos posibles (Δfila, Δcolumna) son:

```
(-2,-1) (-2,+1)
(-1,-2)         (-1,+2)
        [♞]
(+1,-2)         (+1,+2)
(+2,-1) (+2,+1)
```


In [ ]:
# Visualización del tablero y movimientos del caballo
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def dibujar_tablero_con_movimientos(pos, titulo="Movimientos posibles del caballo"):
    """Dibuja un tablero 8x8 mostrando los movimientos legales del caballo."""
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    # Dibujar casillas del tablero
    for i in range(8):
        for j in range(8):
            color = '#F0D9B5' if (i + j) % 2 == 0 else '#B58863'
            ax.add_patch(patches.Rectangle((j, 7 - i), 1, 1, 
                                           facecolor=color, edgecolor='#333'))
    
    # Movimientos posibles
    movimientos = [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]
    fila, col = pos
    
    for df, dc in movimientos:
        nf, nc = fila + df, col + dc
        if 0 <= nf < 8 and 0 <= nc < 8:
            ax.add_patch(patches.Rectangle((nc, 7 - nf), 1, 1,
                                           facecolor='#90EE90', alpha=0.7, edgecolor='#333'))
            ax.plot(nc + 0.5, 7 - nf + 0.5, 'o', color='green', markersize=10)
    
    # Posición del caballo
    ax.text(col + 0.5, 7 - fila + 0.5, '♞', fontsize=30, ha='center', va='center',
            color='#1a1a2e', fontweight='bold')
    
    # Etiquetas
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 8)
    ax.set_xticks(np.arange(0.5, 8.5, 1))
    ax.set_yticks(np.arange(0.5, 8.5, 1))
    ax.set_xticklabels([str(i) for i in range(8)])
    ax.set_yticklabels([str(i) for i in range(7, -1, -1)])
    ax.set_xlabel('Columna')
    ax.set_ylabel('Fila')
    ax.set_title(titulo, fontsize=14, fontweight='bold')
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()

# Mostrar movimientos desde el centro (3,3) y desde una esquina (0,0)
dibujar_tablero_con_movimientos((3, 3), "Movimientos del caballo desde el centro (3,3) — 8 saltos")
dibujar_tablero_con_movimientos((0, 0), "Movimientos del caballo desde la esquina (0,0) — 2 saltos")


## 2. Modelado formal

### 2.1 Estado

El estado se representa como una **tupla** `(fila, columna)` donde `0 ≤ fila, columna ≤ 7`.

- **Hashable:** Las tuplas en Python son hashable, lo que permite usarlas como claves en el
  diccionario de estados visitados de `TreeSearch`.
- **Mínimo:** Solo se almacena la posición actual. No se necesita información adicional (como el
  número de movimiento o las casillas visitadas) porque la variante (a) solo busca llegar a la meta,
  no recorrer todo el tablero.
- **Suficiente:** La posición actual es todo lo necesario para calcular los movimientos legales y
  verificar si se alcanzó la meta.

Si agregáramos el número de movimiento al estado (e.g., `(fila, columna, paso)`), dos situaciones
idénticas del problema se verían como estados distintos para el algoritmo, multiplicando
innecesariamente el espacio de estados y anulando la detección de repetidos.

### 2.2 Operación sucesor

Los **8 desplazamientos en L** del caballo son:

| Δfila | Δcolumna |
|-------|----------|
| -2    | -1       |
| -2    | +1       |
| -1    | -2       |
| -1    | +2       |
| +1    | -2       |
| +1    | +2       |
| +2    | -1       |
| +2    | +1       |

Solo se generan hijos cuyas coordenadas caigan dentro del tablero (`0 ≤ fila, col ≤ 7`).

**Factor de ramificación promedio:** b ≈ 5.25 (8 en el centro, 2 en esquinas, ~4 en bordes).

El `step_cost` es **1** para todos los movimientos (costos uniformes).

### 2.3 Condición de meta

La meta se cumple cuando `nodo_actual.state == nodo_meta.state`, es decir, cuando la
posición del caballo coincide exactamente con la casilla destino.

### 2.4 Heurísticas

#### Heurística 1: Manhattan / 3 (principal)

$$h_1(n) = \frac{|f_n - f_{meta}| + |c_n - c_{meta}|}{3}$$

**¿Por qué es razonable?** Cada salto del caballo avanza exactamente 3 casillas en distancia
Manhattan (2 en un eje + 1 en el otro). Dividir la distancia Manhattan total entre 3 estima el
número mínimo de saltos necesarios.

**¿Es admisible?** Sí. Argumento de relajación: si el caballo pudiera elegir libremente la
dirección de sus componentes (sin restricción de forma de L ni límites de tablero), necesitaría
al menos ⌈Manhattan/3⌉ saltos. Dado que usamos `Manhattan/3` sin techo, nunca sobreestimamos:
h₁(n) ≤ h*(n) para todo n.

#### Heurística 2: Chebyshev / 2 (comparación)

$$h_2(n) = \frac{\max(|f_n - f_{meta}|, |c_n - c_{meta}|)}{2}$$

**¿Por qué es admisible?** Cada salto del caballo avanza como máximo 2 casillas en cualquier
dirección (fila o columna). Dividir la distancia Chebyshev entre 2 da una cota inferior del
número de saltos. Es más débil que h₁ (estima menos), por lo que expandirá más nodos.


## 3. Estimación del tamaño de la búsqueda

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| b (factor de ramificación) | ≈ 5.25 | Promedio entre 2 (esquinas) y 8 (centro) |
| d (profundidad máxima) | 6 | Máximo entre dos casillas cualesquiera en 8×8 |
| Estados distintos | 64 | Tablero 8×8 |
| b^d estimado | 5.25⁶ ≈ 20,972 | Sin detección de repetidos |
| Nodos BFS (con repetidos) | ≤ 64 | Acotado por el total de estados |

### Conclusión de viabilidad

Con solo 64 estados posibles y detección de repetidos, **las cuatro estrategias terminan
prácticamente al instante**. La estimación de b^d ≈ 20,972 es el peor caso sin control de
repetidos, pero `TreeSearch` lo aplica, así que el número real de nodos expandidos estará
muy por debajo.

Esto convierte al problema en un escenario ideal para comparar las estrategias sin preocuparse
por límites de tiempo o memoria.


## 4. Implementación

### 4.1 Importar la librería y dependencias


In [ ]:
import sys
import time
import os

# Agregar la carpeta src al path para importar la libreria baile
sys.path.insert(0, os.path.abspath('src'))
import SimpleSearch as sp

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

print("Librería SimpleSearch cargada correctamente.")
print(f"Python: {sys.version}")


### 4.2 Definición de los cuatro componentes


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MOVIMIENTOS DEL CABALLO
# Los 8 saltos en forma de "L": (Δfila, Δcolumna)
# ═══════════════════════════════════════════════════════════════════
MOVIMIENTOS = [
    (-2, -1), (-2, +1),  # dos arriba, uno a cada lado
    (-1, -2), (-1, +2),  # uno arriba, dos a cada lado
    (+1, -2), (+1, +2),  # uno abajo, dos a cada lado
    (+2, -1), (+2, +1),  # dos abajo, uno a cada lado
]

TAMAÑO = 8  # Tablero 8×8


# ═══════════════════════════════════════════════════════════════════
# OPERACIÓN SUCESOR
# Genera los hijos legales del caballo desde el nodo actual.
# Cada hijo se construye con parent y depth para recuperar el
# camino con getPath(). step_cost=1 (costos uniformes).
# ═══════════════════════════════════════════════════════════════════
def sucesor(nodo):
    """Genera los nodos hijos legales del caballo.
    
    Para cada uno de los 8 saltos en L, verifica que la casilla
    destino esté dentro del tablero. Construye el nodo hijo con
    parent y depth para que getPath() funcione correctamente.
    """
    fila, col = nodo.state
    hijos = []
    for df, dc in MOVIMIENTOS:
        nf, nc = fila + df, col + dc
        # Solo se generan estados legales (dentro del tablero)
        if 0 <= nf < TAMAÑO and 0 <= nc < TAMAÑO:
            nuevo_estado = (nf, nc)
            descripcion = f"({fila},{col})→({nf},{nc})"
            hijos.append(sp.node(
                nuevo_estado,
                parent=nodo,
                depth=nodo.depth + 1,
                op=descripcion,
                step_cost=1  # costo uniforme
            ))
    return hijos


# ═══════════════════════════════════════════════════════════════════
# CONDICIÓN DE META
# Compara la posición actual con la posición destino.
# ═══════════════════════════════════════════════════════════════════
def meta(*nodos):
    """La meta se cumple cuando la posición del caballo coincide
    con la casilla destino.
    
    Recibe dos nodos: nodos[0] es el nodo actual, nodos[1] es el
    nodo meta (goal_state). Se comparan sus estados (tuplas).
    """
    return nodos[0].state == nodos[1].state


# ═══════════════════════════════════════════════════════════════════
# HEURÍSTICAS
# ═══════════════════════════════════════════════════════════════════

def h_manhattan3(nodo, nodo_meta):
    """h₁(n) = Manhattan(n, meta) / 3.
    
    Admisible: cada salto del caballo avanza exactamente 3 en
    distancia Manhattan (2+1), así que dividir entre 3 nunca
    sobreestima el número de saltos necesarios.
    """
    f1, c1 = nodo.state
    f2, c2 = nodo_meta.state
    return (abs(f1 - f2) + abs(c1 - c2)) / 3


def h_chebyshev2(nodo, nodo_meta):
    """h₂(n) = Chebyshev(n, meta) / 2.
    
    Admisible: cada salto del caballo avanza como máximo 2 casillas
    en cualquier dirección, así que dividir entre 2 nunca
    sobreestima. Más débil que h₁.
    """
    f1, c1 = nodo.state
    f2, c2 = nodo_meta.state
    return max(abs(f1 - f2), abs(c1 - c2)) / 2


def h_cero(nodo, nodo_meta):
    """h(n) = 0: búsqueda de costo uniforme (línea base).
    
    Con h=0, A* se convierte en búsqueda de costo uniforme
    (Dijkstra). Sirve como línea base para medir cuánto
    aporta cada heurística.
    """
    return 0


print("Componentes definidos: sucesor, meta, h_manhattan3, h_chebyshev2, h_cero")


### 4.3 Verificación rápida del sucesor


In [ ]:
# Verificar que el sucesor genera los movimientos correctos
nodo_centro = sp.node((3, 3))
nodo_esquina = sp.node((0, 0))

hijos_centro = sucesor(nodo_centro)
hijos_esquina = sucesor(nodo_esquina)

print(f"Desde el centro (3,3): {len(hijos_centro)} movimientos legales")
for h in hijos_centro:
    print(f"  → {h.state}  operación: {h.op}")

print(f"\nDesde la esquina (0,0): {len(hijos_esquina)} movimientos legales")
for h in hijos_esquina:
    print(f"  → {h.state}  operación: {h.op}")

# Factor de ramificación promedio
total_mov = sum(len(sucesor(sp.node((r, c)))) for r in range(8) for c in range(8))
b_promedio = total_mov / 64
print(f"\nFactor de ramificación promedio: {b_promedio:.2f}")


## 5. Experimentos

### 5.1 Definición de instancias

Se definen **3 instancias de dificultad creciente**, todas con inicio en `(0,0)`:

| Instancia | Inicio | Meta | Descripción |
|-----------|--------|------|-------------|
| Fácil | (0, 0) | (1, 2) | Un solo salto |
| Media | (0, 0) | (4, 4) | Centro del tablero |
| Difícil | (0, 0) | (7, 7) | Esquina opuesta |

### 5.2 Configuraciones

Se prueban **5 configuraciones** por instancia:
1. **BFS** — Búsqueda en amplitud
2. **DFS** — Búsqueda en profundidad
3. **A\* (h=0)** — Costo uniforme (línea base)
4. **A\* (h₁)** — Manhattan / 3
5. **A\* (h₂)** — Chebyshev / 2

Todas las corridas usan `max_iter=500000` y se ejecutan en la misma máquina.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# FUNCIÓN DE EXPERIMENTACIÓN
# ═══════════════════════════════════════════════════════════════════

MAX_ITER = 500000

def ejecutar_experimento(inicio_pos, meta_pos, nombre_instancia):
    """Ejecuta las 5 configuraciones para una instancia dada.
    
    Retorna una lista de diccionarios con los resultados.
    """
    configuraciones = [
        ("BFS",        "bfs",  None),
        ("DFS",        "dfs",  None),
        ("A* (h=0)",   "a*",   h_cero),
        ("A* (h₁)",    "a*",   h_manhattan3),
        ("A* (h₂)",    "a*",   h_chebyshev2),
    ]
    
    resultados = []
    nodo_meta = sp.node(meta_pos)
    
    for nombre, estrategia, heuristica in configuraciones:
        # Crear nodo inicio fresco para cada corrida
        inicio = sp.node(inicio_pos)
        
        # Crear el motor de búsqueda
        buscador = sp.TreeSearch(
            inicio, sucesor, meta,
            strategy=estrategia,
            goal_state=nodo_meta,
            heuristic=heuristica
        )
        
        # Ejecutar y medir tiempo
        t0 = time.time()
        resultado = buscador.find(max_iter=MAX_ITER)
        t1 = time.time()
        
        if resultado is not None:
            camino = resultado.getPath()
            longitud = len(camino) - 1  # número de movimientos
            costo = resultado.cost      # g(n)
        else:
            longitud = "N/A"
            costo = "N/A"
        
        resultados.append({
            "Instancia": nombre_instancia,
            "Estrategia": nombre,
            "Nodos expandidos": buscador.iterations,
            "Tiempo (s)": round(t1 - t0, 6),
            "Longitud": longitud,
            "Costo g": costo,
        })
    
    return resultados

print("Función de experimentación definida.")


### 5.3 Ejecución de los experimentos


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EJECUTAR TODOS LOS EXPERIMENTOS
# ═══════════════════════════════════════════════════════════════════

instancias = [
    ((0, 0), (1, 2), "Fácil"),     # ~1 movimiento
    ((0, 0), (4, 4), "Media"),     # ~4 movimientos
    ((0, 0), (7, 7), "Difícil"),   # ~6 movimientos
]

todos_resultados = []

for inicio, meta_pos, nombre in instancias:
    print(f"\n{'='*60}")
    print(f"Instancia: {nombre} | Inicio: {inicio} → Meta: {meta_pos}")
    print(f"{'='*60}")
    
    resultados = ejecutar_experimento(inicio, meta_pos, nombre)
    todos_resultados.extend(resultados)
    
    for r in resultados:
        print(f"  {r['Estrategia']:12s} | Nodos: {r['Nodos expandidos']:6} | "
              f"Tiempo: {r['Tiempo (s)']:.6f}s | Long: {r['Longitud']} | "
              f"Costo: {r['Costo g']}")

print("\n✓ Todos los experimentos completados.")


### 5.4 Tabla de resultados


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# TABLA DE RESULTADOS COMPLETA
# ═══════════════════════════════════════════════════════════════════

df = pd.DataFrame(todos_resultados)
print("\n" + "═"*80)
print("TABLA DE RESULTADOS COMPLETA")
print("═"*80)
print(f"max_iter = {MAX_ITER}")
print(f"Python: {sys.version}")
print()
display(df.style.set_caption("Resultados: Caballo de Ajedrez — Variante (a) — Tablero 8×8"))


### 5.5 Verificación de correctitud (sección 6.1 del proyecto)

Con costos uniformes, **BFS siempre devuelve una solución de longitud mínima**. Si la longitud
de A\* coincide con la de BFS, tenemos evidencia de que la heurística es **admisible**.
Si A\* devuelve un camino más largo, la heurística NO es admisible.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# VERIFICACIÓN: longitud BFS == longitud A*
# ═══════════════════════════════════════════════════════════════════

print("Verificación de correctitud: ¿longitud BFS == longitud A*?")
print("═" * 60)

for nombre_inst in ["Fácil", "Media", "Difícil"]:
    fila_bfs = df[(df['Instancia'] == nombre_inst) & (df['Estrategia'] == 'BFS')]
    fila_a1  = df[(df['Instancia'] == nombre_inst) & (df['Estrategia'] == 'A* (h₁)')]
    fila_a2  = df[(df['Instancia'] == nombre_inst) & (df['Estrategia'] == 'A* (h₂)')]
    
    long_bfs = fila_bfs['Longitud'].values[0]
    long_a1  = fila_a1['Longitud'].values[0]
    long_a2  = fila_a2['Longitud'].values[0]
    
    check1 = "✓" if long_bfs == long_a1 else "✗ ¡NO COINCIDE!"
    check2 = "✓" if long_bfs == long_a2 else "✗ ¡NO COINCIDE!"
    
    print(f"\n  Instancia '{nombre_inst}':")
    print(f"    BFS: {long_bfs} movimientos")
    print(f"    A*(h₁): {long_a1} movimientos → {check1}")
    print(f"    A*(h₂): {long_a2} movimientos → {check2}")

# Verificación automática
todas_coinciden = True
for nombre_inst in ["Fácil", "Media", "Difícil"]:
    bfs_l = df[(df['Instancia'] == nombre_inst) & (df['Estrategia'] == 'BFS')]['Longitud'].values[0]
    for h_name in ['A* (h₁)', 'A* (h₂)']:
        a_l = df[(df['Instancia'] == nombre_inst) & (df['Estrategia'] == h_name)]['Longitud'].values[0]
        if bfs_l != a_l:
            todas_coinciden = False

if todas_coinciden:
    print("\n✓ TODAS las longitudes coinciden → ambas heurísticas son ADMISIBLES.")
else:
    print("\n✗ ALGUNA longitud no coincide → revisar la heurística correspondiente.")


### 5.6 Visualización de los caminos encontrados


In [ ]:
def dibujar_camino(inicio_pos, meta_pos, nombre_instancia):
    """Ejecuta BFS y dibuja el camino óptimo en el tablero."""
    inicio = sp.node(inicio_pos)
    nodo_meta = sp.node(meta_pos)
    
    buscador = sp.TreeSearch(inicio, sucesor, meta,
                             strategy="bfs", goal_state=nodo_meta)
    resultado = buscador.find(max_iter=MAX_ITER)
    
    if resultado is None:
        print(f"No se encontró solución para {nombre_instancia}")
        return
    
    camino = resultado.getPath()
    posiciones = [paso[0] for paso in camino]
    
    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    
    # Dibujar casillas del tablero
    for i in range(8):
        for j in range(8):
            color = '#F0D9B5' if (i + j) % 2 == 0 else '#B58863'
            ax.add_patch(patches.Rectangle((j, 7 - i), 1, 1,
                                           facecolor=color, edgecolor='#333'))
    
    # Marcar casillas del camino
    for idx, (f, c) in enumerate(posiciones):
        if idx == 0:
            color_casilla = '#4CAF50'  # Verde: inicio
        elif idx == len(posiciones) - 1:
            color_casilla = '#F44336'  # Rojo: meta
        else:
            color_casilla = '#64B5F6'  # Azul: intermedio
        
        ax.add_patch(patches.Rectangle((c, 7 - f), 1, 1,
                                       facecolor=color_casilla, alpha=0.7,
                                       edgecolor='#333'))
    
    # Dibujar flechas del camino
    for idx in range(len(posiciones) - 1):
        f1, c1 = posiciones[idx]
        f2, c2 = posiciones[idx + 1]
        ax.annotate('', xy=(c2 + 0.5, 7 - f2 + 0.5),
                    xytext=(c1 + 0.5, 7 - f1 + 0.5),
                    arrowprops=dict(arrowstyle='->', color='#1a1a2e',
                                   lw=2.5, connectionstyle='arc3,rad=0.2'))
    
    # Numerar los pasos
    for idx, (f, c) in enumerate(posiciones):
        ax.text(c + 0.5, 7 - f + 0.5, str(idx), fontsize=16,
                ha='center', va='center', fontweight='bold',
                color='white',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#1a1a2e', alpha=0.8))
    
    # Etiquetas
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 8)
    ax.set_xticks(np.arange(0.5, 8.5, 1))
    ax.set_yticks(np.arange(0.5, 8.5, 1))
    ax.set_xticklabels([str(i) for i in range(8)])
    ax.set_yticklabels([str(i) for i in range(7, -1, -1)])
    ax.set_xlabel('Columna', fontsize=12)
    ax.set_ylabel('Fila', fontsize=12)
    longitud = len(posiciones) - 1
    ax.set_title(f"{nombre_instancia}: {inicio_pos} → {meta_pos}  "
                 f"({longitud} movimientos)",
                 fontsize=14, fontweight='bold')
    ax.set_aspect('equal')
    
    # Leyenda
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='s', color='w', markerfacecolor='#4CAF50',
               markersize=15, label='Inicio'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='#F44336',
               markersize=15, label='Meta'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='#64B5F6',
               markersize=15, label='Intermedio'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Imprimir el camino
    print(f"Camino ({longitud} movimientos):")
    for i, (estado, op, prof) in enumerate(camino):
        if i == 0:
            print(f"  Paso {i}: Inicio en {estado}")
        else:
            print(f"  Paso {i}: {op}  (profundidad {prof})")


# Visualizar los caminos óptimos de las 3 instancias
for inicio, meta_pos, nombre in instancias:
    dibujar_camino(inicio, meta_pos, nombre)
    print()


### 5.7 Gráficas comparativas


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# GRÁFICA: Nodos expandidos por estrategia e instancia
# ═══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica 1: Nodos expandidos
ax1 = axes[0]
instancias_nombres = ["Fácil", "Media", "Difícil"]
estrategias = ["BFS", "DFS", "A* (h=0)", "A* (h₁)", "A* (h₂)"]
colores = ['#2196F3', '#FF9800', '#9E9E9E', '#4CAF50', '#9C27B0']

x = np.arange(len(instancias_nombres))
width = 0.15

for i, (est, color) in enumerate(zip(estrategias, colores)):
    nodos = []
    for inst in instancias_nombres:
        val = df[(df['Instancia'] == inst) & (df['Estrategia'] == est)]['Nodos expandidos'].values[0]
        nodos.append(val)
    ax1.bar(x + i * width, nodos, width, label=est, color=color, edgecolor='#333', alpha=0.85)

ax1.set_xlabel('Instancia', fontsize=12)
ax1.set_ylabel('Nodos expandidos', fontsize=12)
ax1.set_title('Nodos expandidos por estrategia', fontsize=14, fontweight='bold')
ax1.set_xticks(x + width * 2)
ax1.set_xticklabels(instancias_nombres)
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# Gráfica 2: Longitud de la solución
ax2 = axes[1]
for i, (est, color) in enumerate(zip(estrategias, colores)):
    longitudes = []
    for inst in instancias_nombres:
        val = df[(df['Instancia'] == inst) & (df['Estrategia'] == est)]['Longitud'].values[0]
        if isinstance(val, str):
            val = 0
        longitudes.append(val)
    ax2.bar(x + i * width, longitudes, width, label=est, color=color, edgecolor='#333', alpha=0.85)

ax2.set_xlabel('Instancia', fontsize=12)
ax2.set_ylabel('Longitud del camino', fontsize=12)
ax2.set_title('Longitud de la solución por estrategia', fontsize=14, fontweight='bold')
ax2.set_xticks(x + width * 2)
ax2.set_xticklabels(instancias_nombres)
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Análisis de la heurística

### Pregunta 1: ¿Qué mide tu heurística y por qué es razonable?

**h₁(n) = Manhattan(n, meta) / 3** mide la estimación del número mínimo de saltos del caballo
necesarios para ir del estado actual al estado meta. Es razonable porque cada salto del caballo
siempre avanza exactamente **3 casillas en distancia Manhattan** (2 casillas en un eje + 1 casilla
en el otro). Dividir la distancia Manhattan total entre 3 da una buena aproximación del esfuerzo
restante.

### Pregunta 2: ¿Es admisible?

**Sí, h₁ es admisible.** 

**Argumento de relajación:** Si relajamos el problema permitiendo que el caballo se mueva sin
restricción de forma de L (pero manteniendo que cada salto cuesta 1 y avanza Manhattan=3), el
problema relajado se resuelve en exactamente ⌈Manhattan/3⌉ pasos. Dado que:

1. Usamos `Manhattan/3` **sin techo** (redondeado hacia abajo implícitamente), por lo que
   h₁(n) ≤ ⌈Manhattan/3⌉ ≤ h*(n)
2. El problema real no puede ser más fácil que el relajado

Por lo tanto, h₁(n) ≤ h*(n) para todo n, y la heurística nunca sobreestima.

**Verificación empírica:** En la sección 5.5 se comprobó que la longitud de A*(h₁) coincide con
la de BFS para las tres instancias, lo cual es consistente con la admisibilidad.

### Pregunta 3: ¿Cuánto ayuda?


In [ ]:
# Comparar A*(h₁) vs A*(h=0) en nodos expandidos
print("Comparación: A*(h₁) vs A*(h=0) — ¿cuánto ayuda la heurística?")
print("═" * 65)

for inst in ["Fácil", "Media", "Difícil"]:
    n_h0 = df[(df['Instancia'] == inst) & (df['Estrategia'] == 'A* (h=0)')]['Nodos expandidos'].values[0]
    n_h1 = df[(df['Instancia'] == inst) & (df['Estrategia'] == 'A* (h₁)')]['Nodos expandidos'].values[0]
    n_h2 = df[(df['Instancia'] == inst) & (df['Estrategia'] == 'A* (h₂)')]['Nodos expandidos'].values[0]
    
    if n_h1 > 0:
        reduccion_h1 = ((n_h0 - n_h1) / n_h0) * 100 if n_h0 > 0 else 0
    else:
        reduccion_h1 = 100.0
    
    if n_h2 > 0:
        reduccion_h2 = ((n_h0 - n_h2) / n_h0) * 100 if n_h0 > 0 else 0
    else:
        reduccion_h2 = 100.0
    
    print(f"\n  Instancia '{inst}':")
    print(f"    A*(h=0): {n_h0:5d} nodos")
    print(f"    A*(h₁):  {n_h1:5d} nodos  → reducción {reduccion_h1:.1f}%")
    print(f"    A*(h₂):  {n_h2:5d} nodos  → reducción {reduccion_h2:.1f}%")

print("\nConclusión: h₁ (Manhattan/3) reduce significativamente los nodos expandidos")
print("respecto a h=0. h₂ (Chebyshev/2) también reduce, pero menos que h₁")
print("porque es una heurística más débil (estima menos).")


### Pregunta 4: Segunda heurística comparada

La segunda heurística es **h₂(n) = Chebyshev(n, meta) / 2**, donde la distancia Chebyshev
es el máximo de las distancias absolutas en fila y columna.

**¿Es admisible?** Sí. Cada salto del caballo avanza como máximo 2 casillas en cualquier
dirección (fila o columna). Dividir la distancia Chebyshev entre 2 da una cota inferior del
número de saltos necesarios, por lo que nunca sobreestima.

**¿Es más fuerte o más débil que h₁?** Es **más débil**. Para cualquier estado:

$$h_2(n) = \frac{\max(|\Delta f|, |\Delta c|)}{2} \leq \frac{|\Delta f| + |\Delta c|}{3} = h_1(n)$$

No siempre se cumple esta desigualdad (depende de la proporción Δf/Δc), pero en la
mayoría de los casos h₁ ≥ h₂, lo que significa que h₁ es más informativa y guía la búsqueda
de forma más eficiente, expandiendo menos nodos.

Los resultados experimentales confirman que A*(h₁) expande menos nodos que A*(h₂)
para las instancias probadas.


### Comprobación exhaustiva de admisibilidad (sección 7.1)

Como el espacio de estados es pequeño (64 casillas), podemos calcular h\* exactamente
para todas las casillas y verificar que h(n) ≤ h\*(n) en **todos** los estados.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# COMPROBACIÓN EXHAUSTIVA DE ADMISIBILIDAD
# Calcular h* desde cada casilla a la meta (7,7) usando BFS,
# y verificar que h₁(n) ≤ h*(n) y h₂(n) ≤ h*(n) para todo n.
# ═══════════════════════════════════════════════════════════════════

meta_test = (7, 7)
nodo_meta_test = sp.node(meta_test)

print(f"Comprobación exhaustiva de admisibilidad respecto a meta={meta_test}")
print("═" * 65)

# Calcular h* para cada casilla con BFS
h_star = {}
for f in range(8):
    for c in range(8):
        inicio = sp.node((f, c))
        b = sp.TreeSearch(inicio, sucesor, meta, strategy="bfs",
                          goal_state=nodo_meta_test)
        r = b.find(max_iter=MAX_ITER)
        if r is not None:
            h_star[(f, c)] = len(r.getPath()) - 1
        else:
            h_star[(f, c)] = float('inf')

# Verificar admisibilidad
admisible_h1 = True
admisible_h2 = True
violaciones_h1 = []
violaciones_h2 = []

for f in range(8):
    for c in range(8):
        n = sp.node((f, c))
        h1_val = h_manhattan3(n, nodo_meta_test)
        h2_val = h_chebyshev2(n, nodo_meta_test)
        hstar_val = h_star[(f, c)]
        
        if h1_val > hstar_val:
            admisible_h1 = False
            violaciones_h1.append((f, c, h1_val, hstar_val))
        if h2_val > hstar_val:
            admisible_h2 = False
            violaciones_h2.append((f, c, h2_val, hstar_val))

print(f"\n  h₁ (Manhattan/3): {'ADMISIBLE ✓' if admisible_h1 else 'NO ADMISIBLE ✗'}")
if violaciones_h1:
    for v in violaciones_h1:
        print(f"    Violación en {v[0:2]}: h₁={v[2]:.2f} > h*={v[3]}")

print(f"  h₂ (Chebyshev/2): {'ADMISIBLE ✓' if admisible_h2 else 'NO ADMISIBLE ✗'}")
if violaciones_h2:
    for v in violaciones_h2:
        print(f"    Violación en {v[0:2]}: h₂={v[2]:.2f} > h*={v[3]}")

# Mostrar tabla h* vs h₁ vs h₂ para algunas casillas
print("\n\nMuestra de valores h* vs h₁ vs h₂:")
print(f"{'Casilla':>10s} {'h*':>5s} {'h₁':>8s} {'h₂':>8s} {'h₁≤h*':>8s} {'h₂≤h*':>8s}")
print("-" * 50)
casillas_muestra = [(0,0), (0,7), (3,3), (7,0), (6,5), (1,1), (7,6)]
for (f, c) in casillas_muestra:
    n = sp.node((f, c))
    h1 = h_manhattan3(n, nodo_meta_test)
    h2 = h_chebyshev2(n, nodo_meta_test)
    hs = h_star[(f, c)]
    print(f"  ({f},{c})     {hs:5d} {h1:8.2f} {h2:8.2f} {'✓':>8s} {'✓':>8s}")


## 7. Comparación: estimación teórica vs resultados experimentales


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# COMPARACIÓN DE LA ESTIMACIÓN PREVIA CON LA REALIDAD
# ═══════════════════════════════════════════════════════════════════

print("Comparación: estimación teórica vs nodos realmente expandidos")
print("═" * 65)
print()
print("Estimación previa (sección 5 de la propuesta):")
print("  b ≈ 5.25, d_max = 6")
print(f"  b^d = 5.25^6 ≈ {5.25**6:,.0f} nodos (sin detección de repetidos)")
print(f"  Con detección de repetidos: ≤ 64 estados")
print()

# Real factor de ramificación
print(f"Factor de ramificación real medido: {b_promedio:.2f}")
print()

print("Nodos realmente expandidos (instancia Difícil):")
for est in estrategias:
    val = df[(df['Instancia'] == 'Difícil') & (df['Estrategia'] == est)]['Nodos expandidos'].values[0]
    print(f"  {est:12s}: {val:6d} nodos")

print("\nAnálisis:")
print("  La estimación teórica de b^d ≈ 20,972 era el peor caso sin detección")
print("  de repetidos. Con detección de repetidos (que TreeSearch aplica),")
print("  BFS y A* no pueden expandir más de 64 nodos distintos.")
print("  Los valores reales están muy por debajo de la estimación teórica,")
print("  lo que confirma que la detección de repetidos es efectiva.")


## 8. Conclusiones

### ¿Qué estrategia ganó y por qué?

1. **A\*(h₁)** (Manhattan/3) es la estrategia ganadora en nodos expandidos. La heurística
   guía la búsqueda directamente hacia la meta, evitando explorar casillas irrelevantes.

2. **BFS** garantiza encontrar el camino óptimo (de longitud mínima) y sirve como referencia
   de correctitud. Expande más nodos que A*(h₁) pero menos que DFS para las instancias difíciles.

3. **DFS** puede encontrar una solución rápidamente (pocos nodos expandidos), pero el camino
   encontrado **no es necesariamente óptimo**: puede ser mucho más largo que el mínimo.
   Esto se evidencia en la columna "Longitud", donde DFS frecuentemente devuelve caminos
   más largos que BFS.

4. **A\*(h=0)** (costo uniforme) se comporta como BFS con costos uniformes, expandiendo
   prácticamente los mismos nodos. Sirve como línea base para demostrar que la heurística
   sí aporta información útil.

5. **A\*(h₂)** (Chebyshev/2) reduce nodos respecto a h=0, pero menos que h₁, confirmando
   que una heurística más informativa guía mejor la búsqueda.

### ¿Por qué DFS expande menos nodos pero encuentra caminos más largos?

DFS sigue un camino tan profundo como puede antes de retroceder. En este problema,
es posible que alcance la meta sin explorar muchas alternativas, pero por una ruta
subóptima. Expandir **menos nodos no significa encontrar mejor solución**.

### ¿Qué haríamos distinto?

Si tuviéramos más tiempo, exploraríamos heurísticas más finas como la fórmula exacta
del caballo en tablero infinito, que da el número exacto de saltos en muchos casos y
sería una heurística perfecta (h = h*) para casillas alejadas de los bordes.

### Sobre `step_cost`

No se necesita `step_cost` distinto de 1 porque todos los movimientos del caballo tienen
el mismo costo (un salto = un movimiento). Con costos uniformes, la longitud del camino
coincide con el costo g(n).

### Aportaciones de los integrantes

*   **Ivan Montoya:** Modelado formal de los componentes, justificación de las heurísticas admisibles y comprobación exhaustiva del espacio de estados.
*   **Gustavo Perez:** Implementación del código de búsqueda, estructuración de los experimentos y visualización gráfica de los caminos y métricas.
*   **Johan Cruz:** Análisis del rendimiento de las estrategias (A* vs DFS vs BFS), interpretación de la longitud de caminos e integración del reporte final.


## 9. Referencias

1. Russell, S. & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.).
   Pearson. Capítulos 3-4: Búsqueda en espacios de estados.

2. Ortiz-Bejar, J. (2018). *baile: Basic Artificial Intelligence Library for Education*.
   Repositorio: [https://github.com/kyriox/baile](https://github.com/kyriox/baile)

3. Wikipedia. *Knight's tour*.
   [https://en.wikipedia.org/wiki/Knight%27s_tour](https://en.wikipedia.org/wiki/Knight%27s_tour)

4. Wikipedia. *Admissible heuristic*.
   [https://en.wikipedia.org/wiki/Admissible_heuristic](https://en.wikipedia.org/wiki/Admissible_heuristic)
